In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split 
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, recall_score, precision_score,f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier

In [ ]:
X = pd.read_csv("/kaggle/input/titanic/train.csv")
X.head()

# EDA

In [ ]:
X["Survived"].value_counts()

We can see that the target variable is slightly unbalanced and there's no missing value hidden here.

In [ ]:
X.columns

In [ ]:
X.info()

We can drop the Cabin column for two reasons. First, it contains a lot of missing values. Second, the existing values are not directly usable and would require additional feature engineering. Therefore, in the first version, I will not include it.

In [ ]:
X.drop(columns = ["Cabin","Name","Ticket"],inplace = True)

In [ ]:
plt.hist(X[X["Survived"] == 0]["Age"],bins=30, color = "red", alpha = 0.5, label = "Did not survive")
plt.hist(X[X["Survived"] == 1]["Age"],bins=30, color = "green",alpha = 0.5, label = "Survived")
plt.legend()
plt.show()

This visualization shows that among young passengers (under 10), there is a clear peak in the number of survivors. This is likely because younger passengers were given priority during the evacuation. For passengers in their 20s and 30s, the trend is reversed, with a lower survival rate. Although the data is not scaled, it still provides a general idea of the overall distribution.

In [ ]:
X["Sex"].value_counts()

In [ ]:
plt.figure(figsize = (10,3))
plt.subplot(1,2,1)
plt.hist(X[(X["Survived"] == 0) & (X["Sex"] == "female")]["Age"],bins=30, color = "red", alpha = 0.5, label = "Did not survive")
plt.hist(X[(X["Survived"] == 1) & (X["Sex"] == "female")]["Age"],bins=30, color = "green", alpha = 0.5, label = "Survived")
plt.title("Age Distribution - Females")
plt.xlabel("Age")
plt.ylabel("Number of Passengers")
plt.legend()
plt.subplot(1,2,2)
plt.hist(X[(X["Survived"] == 0) & (X["Sex"] == "male")]["Age"],bins=30, color = "red", alpha = 0.5, label = "Did not survive")
plt.hist(X[(X["Survived"] == 1) & (X["Sex"] == "male")]["Age"],bins=30, color = "green", alpha = 0.5, label = "Survived")
plt.title("Age Distribution - Females")
plt.xlabel("Age")
plt.ylabel("Number of Passengers")
plt.legend()
plt.show()

Based on these visualizations, we can see that sex is clearly a strong predictor of passenger survival. We will also drop rows with missing age values, as age is another strong predictor.

In [ ]:
X["Age"] = X["Age"].fillna(X["Age"].median())
X["Fare"] = X["Fare"].fillna(X["Fare"].median())
X["Embarked"] = X["Embarked"].fillna(X["Embarked"].mode()[0])

In [ ]:
X.info()

In [ ]:
sns.boxplot(X["Fare"]) 
plt.title("Fare Distribution")       
plt.xlabel("Fare")                   
plt.ylabel("Number of Passengers")   
plt.show()

In [ ]:
X["Fare_log"] = np.log1p(X["Fare"])

In [ ]:
numerical_columns = list(X.select_dtypes(include = "number").columns)
numerical_columns[1], numerical_columns[7] = numerical_columns[7], numerical_columns[1]

In [ ]:
matrix = X[numerical_columns].corr()
sns.heatmap(matrix, annot = True, cmap="coolwarm", fmt = ".2f")

In [ ]:
X.drop(columns = ["Fare","PassengerId"],inplace = True)

It's interesting to see that the ticket price is slightly correlated with the survival column. This might lead us to think that the more expensive the ticket, the higher the chance of survival. This feature is therefore quite interesting since ticket price is related to the passenger class, which is inversely correlated with the survival rate. In other words, expensive tickets usually mean first-class passengers, and class is inversely correlated with survival. It's important to keep these relationships in mind because they can introduce a sort of redundancy in the features. Indeed, if we give too much importance to multiple correlated features, we might end up emphasizing the same pattern multiple times, which could lead to overfitting.

# Preprocessing

In [ ]:
X, y = X.drop(columns = ["Survived"]), X["Survived"]

In [ ]:
# Label encoding 
categorical_columns = X.select_dtypes(exclude = "number").columns
label_encoders = {}
for col in categorical_columns : 
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le
X.head(5)

In [ ]:
sc = StandardScaler()
sc.fit(X)
X_scaled = sc.transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size = 0.2, random_state = 42)

In [ ]:
features = X.columns

# Training

## Baseline : Logistic Reg

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
logr = LogisticRegression()
logr.fit(X_train, y_train)

In [ ]:
predicted = logr.predict(X_test)

In [ ]:
cm = confusion_matrix(y_test, predicted)
disp = ConfusionMatrixDisplay(confusion_matrix = cm)
disp.plot()

In [ ]:
print("recall_score = ",recall_score(y_test, predicted))
print("precision_score = ",precision_score(y_test, predicted))
print("f1_score = ",f1_score(y_test, predicted))

## Feature engineering

In [ ]:
X.columns

In [ ]:
X["IsAlone"] = X.apply(lambda x : 0 if (x["SibSp"] + x["Parch"]) > 0 else 1, axis = 1)
X["isYoung"] = X.apply(lambda x : 1 if (x["Age"] < 10) else 0, axis = 1)
features = X.columns

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

## Model development: Decision Tree

In [ ]:
dtree = DecisionTreeClassifier(max_depth = 3, random_state = 42)
dtree.fit(X_train, y_train)

In [ ]:
predicted_tree = dtree.predict(X_test)

In [ ]:
cm_tree = confusion_matrix(y_test, predicted_tree)
disp_tree = ConfusionMatrixDisplay(confusion_matrix = cm_tree)
disp_tree.plot()

In [ ]:
print("recall_score = ",recall_score(y_test, predicted_tree))
print("precision_score = ",precision_score(y_test, predicted_tree))
print("f1_score = ",f1_score(y_test, predicted_tree))

In [ ]:
tree.plot_tree(dtree, feature_names=features, )
plt.show()

## Model development: Random Forest

In [ ]:
X_train

In [ ]:
rf = RandomForestClassifier( )
rf.fit(X_train, y_train)

In [ ]:
predicted_rf = rf.predict(X_test)

In [ ]:
cm_tree = confusion_matrix(y_test, predicted_rf)
disp_tree = ConfusionMatrixDisplay(confusion_matrix = cm_tree)
disp_tree.plot()

In [ ]:
print("recall_score = ",recall_score(y_test, predicted_rf))
print("precision_score = ",precision_score(y_test, predicted_rf))
print("f1_score = ",f1_score(y_test, predicted_rf))

## Model development: LGBM Classifier

In [ ]:
from lightgbm import LGBMClassifier, early_stopping

In [ ]:
lgb = LGBMClassifier()

In [ ]:
# %%
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [5, 7, 10],
    'num_leaves': [31, 50, 70],
    'min_child_samples': [10, 20, 30],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
}

lgb = LGBMClassifier(
    device='gpu',
    random_state=42,
    verbose=-1
)

grid_search = GridSearchCV(
    lgb, 
    param_grid, 
    cv=2,  # Minimum obligatoire
    scoring='f1',
    n_jobs=1,
    verbose=0
)

grid_search.fit(X_train, y_train)

# %%
best_lgb = grid_search.best_estimator_
predicted_lgb = best_lgb.predict(X_test)

print(f"Meilleurs params: {grid_search.best_params_}")
print(f"F1-score: {f1_score(y_test, predicted_lgb):.4f}")

In [ ]:
lgb.fit(X_train, y_train)

In [ ]:
best_lgb = grid_search.best_estimator_

In [ ]:
predicted_lgb = best_lgb.predict(X_test)

print(f"Best params: {grid_search.best_params_}")

In [ ]:
cm_tree = confusion_matrix(y_test, predicted_lgb)
disp_tree = ConfusionMatrixDisplay(confusion_matrix = cm_tree)
disp_tree.plot()

In [ ]:
print("recall_score = ",recall_score(y_test, predicted_lgb))
print("precision_score = ",precision_score(y_test, predicted_lgb))
print("f1_score = ",f1_score(y_test, predicted_lgb))

# Final submission

In [ ]:
X_val = pd.read_csv("/kaggle/input/titanic/test.csv")

In [ ]:
X_val["Age"] = X_val["Age"].fillna(X_val["Age"].median())
X_val["Fare"] = X_val["Fare"].fillna(X_val["Fare"].median())
X_val["Embarked"] = X_val["Embarked"].fillna(X_val["Embarked"].mode()[0])
X_val["Fare_log"] = np.log1p(X_val["Fare"])
PassengerId = X_val["PassengerId"]
X_val.drop(columns = ["Cabin","Name","Ticket","Fare","PassengerId"],inplace = True)

In [ ]:
X_val["IsAlone"] = X_val.apply(lambda x : 0 if (x["SibSp"] + x["Parch"]) > 0 else 1, axis = 1)
X_val["isYoung"] = X_val.apply(lambda x : 1 if (x["Age"] < 10) else 1, axis = 1)

In [ ]:
# Label encoding 
for col in categorical_columns : 
    X_val[col] = label_encoders[col].transform(X_val[col])
X_val

In [ ]:
final_prediction = best_lgb.predict(X_val)

In [ ]:
result_df = pd.DataFrame({
    "PassengerId":PassengerId,
    "Survived":final_prediction
})
result_df.head()

In [ ]:
result_df.to_csv("my_submission.csv", index = False)